In [2]:
import pandas as pd
import json
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
print("Libraries imported!")

Libraries imported!


In [3]:
df = pd.read_csv('data/arxiv_ai_papers.csv')
print(f"Loaded {len(df)} papers")

Loaded 3000 papers


In [4]:
def chunk_text(text, chunk_size=512, overlap=50):
    words = text.split()
    chunks = []
    start = 0
    
    while start < len(words):
        end = start + chunk_size
        chunk = ' '.join(words[start:end])
        chunks.append(chunk)
        start = end - overlap
        
        if start >= len(words):
            break
    
    return chunks

all_chunks = []
chunk_id = 0

for idx, row in df.iterrows():
    chunks = chunk_text(row['abstract'], chunk_size=512, overlap=50)
    
    for chunk in chunks:
        all_chunks.append({
            'chunk_id': chunk_id,
            'paper_id': row['id'],
            'text': chunk
        })
        chunk_id += 1

chunks_df = pd.DataFrame(all_chunks)
print(f"Total chunks: {len(chunks_df)}")
print(f"Sample chunk:")
print(chunks_df['text'][0][:200])

Total chunks: 3725
Sample chunk:
additive models play an important role in semiparametric statistics . this paper gives learning rates for regularized kernel based methods for additive models . these learning rates compare favourably


In [5]:
os.makedirs('data', exist_ok=True)
chunks_df.to_csv('data/chunks_512.csv', index=False)
print(f"Saved {len(chunks_df)} chunks to data/chunks_512.csv")

Saved 3725 chunks to data/chunks_512.csv


In [6]:
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

def generate_questions(abstract, num_questions=2):
    prompt = f"""Based on this research paper abstract, generate {num_questions} specific questions that can be answered from the abstract.

Abstract: {abstract}

Return ONLY a JSON array of questions like this:
["question 1", "question 2"]

No other text, just the JSON array."""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    
    try:
        questions = json.loads(response.choices[0].message.content)
        return questions
    except:
        return []

# Generate from 10 random papers
sample_papers = df.sample(10, random_state=42)
test_questions = []

print("Generating test questions...")

for idx, row in sample_papers.iterrows():
    questions = generate_questions(row['abstract'])
    for q in questions:
        test_questions.append({
            'question': q,
            'source_abstract': row['abstract'],
            'paper_id': row['id']
        })
    print(f"✓ Paper {row['id']} done")

print(f"\nTotal questions generated: {len(test_questions)}")

Generating test questions...
✓ Paper 1801 done
✓ Paper 1190 done
✓ Paper 1817 done
✓ Paper 251 done
✓ Paper 2505 done
✓ Paper 1117 done
✓ Paper 1411 done
✓ Paper 2113 done
✓ Paper 408 done
✓ Paper 2579 done

Total questions generated: 20


In [7]:
# Show all questions
print("=== YOUR TEST QUESTIONS ===\n")
for i, item in enumerate(test_questions):
    print(f"{i+1}. {item['question']}")

# Save to file
with open('data/test_questions.json', 'w') as f:
    json.dump(test_questions, f, indent=2)

print(f"\nSaved to data/test_questions.json")

=== YOUR TEST QUESTIONS ===

1. What type of system is being analyzed in the paper for the mean resolvent using a polymer expansion?
2. What is the significance of the asymptotic expansion for the density of states in the context of the research paper?
3. What is the asymptotic long-time equivalence being referred to in the context of the paper?
4. How does rescaling and respeeding of a renewal process lead to the space-time fractional diffusion equation?
5. What types of functions can the hybrid method be applied to for asymptotic coefficient extraction?
6. What are some examples of infinite product generating functions to which the hybrid method can be applied?
7. What is the importance of optical spectroscopy in determining physical parameters of giant radio galaxies?
8. Why is the SALT instrument considered the best for this task?
9. What is the role of minimal polynomials in computations involving zero-dimensional ideals in polynomial rings?
10. How can minimal polynomials be comp